In [ ]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df, convert_col
import pickle
import cudf

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/sahilbhatia/pandax/dias_notebooks/nickwan/creating-player-stats-using-tracking-data/src/small_bench/checkpoints/post_cell_1.pickle

In [ ]:
%%cudf.pandas.profile
### cell 2 ###

# 1) pull out the row‐indices where we see a ball snap (GPU array)
snap_rows = data.loc[data["event"] == "ball_snap", "frameId"]
snap_idxs = snap_rows.index.values      # cudf array of ints

# 2) build a length‐6 offset vector on the GPU
offsets = pd.Series(range(6)).values     # cudf array [0,1,2,3,4,5]

# 3) broadcast add to get a 2D array of shape (num_snaps, 6), then transpose and flatten in offset‐major order
#    so we reproduce the original x = [(idxs + off).tolist() for off in range(6)] flatten order
grid = snap_idxs.reshape(-1, 1) + offsets   # shape (num_snaps, 6)
all_idxs = grid.T.reshape(-1)               # shape (6 * num_snaps,)

# 4) one final .tolist() (single CPU round‐trip) and reindex to pick rows (with duplicates) in the same order as before
df = data.reindex(all_idxs.tolist())
